# Compilar Data Consumer APK
Ejecuta todas las celdas en orden (1 a 6). Al final se descarga el APK.

**Tiempo estimado total:** 10-15 minutos la primera vez.

## 1. Instalar dependencias del sistema

In [ ]:
import time
_t = time.time()

print('=' * 50)
print('PASO 1: Instalando dependencias del sistema...')
print('=' * 50)

print('[1/3] Actualizando paquetes del sistema...')
!sudo apt-get update -qq 2>/dev/null
print('      OK')

print('[2/3] Instalando build tools, Java, cmake...')
!sudo apt-get install -y -qq build-essential git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev 2>/dev/null | tail -1
print('      OK')

print('[3/3] Instalando buildozer y cython...')
!pip install -q buildozer cython 2>/dev/null
print('      OK')

print()
print(f'PASO 1 COMPLETADO en {time.time() - _t:.0f} segundos')

## 2. Crear directorio del proyecto

In [ ]:
import os

print('=' * 50)
print('PASO 2: Creando directorio del proyecto...')
print('=' * 50)

os.makedirs('/content/dataconsumer', exist_ok=True)
os.chdir('/content/dataconsumer')

print(f'Directorio creado: {os.getcwd()}')
print()
print('PASO 2 COMPLETADO')

## 3. Crear codigo fuente (main.py)

In [ ]:
%%writefile main.py
"""
Data Consumer - Consume datos moviles sin descargar archivos al dispositivo.
Lee datos en memoria y los descarta. Tambien sube datos para consumir en ambas direcciones.
Muestra velocidad, consumo acumulado y permite definir un objetivo en MB o GB.
"""

import os
import ssl
import threading
import time
import urllib.request
import urllib.error

try:
    import certifi
    SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
except Exception:
    SSL_CONTEXT = ssl.create_default_context()

from kivy.app import App
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.label import Label
from kivy.uix.button import Button
from kivy.uix.textinput import TextInput
from kivy.uix.spinner import Spinner
from kivy.uix.progressbar import ProgressBar
from kivy.clock import Clock
from kivy.core.window import Window
from kivy.metrics import dp

DOWNLOAD_URLS = [
    "https://speedtest.tele2.net/100MB.zip",
    "https://speedtest.tele2.net/10MB.zip",
    "https://proof.ovh.net/files/100Mb.dat",
    "https://proof.ovh.net/files/10Mb.dat",
    "https://ipv4.download.thinkbroadband.com/100MB.zip",
    "https://ipv4.download.thinkbroadband.com/10MB.zip",
]

UPLOAD_URLS = [
    "https://speedtest.tele2.net/upload.php",
    "https://speed.hetzner.de/upload.php",
]

CHUNK_SIZE = 131072
UPLOAD_BLOCK_SIZE = 1_048_576


class DataConsumerApp(App):
    def build(self):
        self.title = "Data Consumer"
        self.total_bytes = 0
        self.target_bytes = 0
        self.running = False
        self.speed_bytes = 0
        self.speed_tracker_bytes = 0
        self.lock = threading.Lock()

        Window.clearcolor = (0.08, 0.08, 0.11, 1)

        root = BoxLayout(orientation="vertical", padding=dp(16), spacing=dp(8))

        root.add_widget(Label(
            text="[b]DATA CONSUMER[/b]",
            markup=True, font_size=dp(24),
            size_hint_y=None, height=dp(40),
            color=(0.3, 0.75, 1, 1),
        ))

        root.add_widget(Label(
            text="Consume datos sin guardar archivos",
            font_size=dp(12),
            size_hint_y=None, height=dp(20),
            color=(0.5, 0.5, 0.5, 1),
        ))

        self.speed_label = Label(
            text="0.00 Mbps",
            font_size=dp(34), bold=True,
            size_hint_y=None, height=dp(50),
            color=(0.2, 0.95, 0.4, 1),
        )
        root.add_widget(self.speed_label)

        self.consumed_label = Label(
            text="Consumido: 0.00 MB",
            font_size=dp(20),
            size_hint_y=None, height=dp(38),
            color=(1, 1, 1, 1),
        )
        root.add_widget(self.consumed_label)

        self.target_display = Label(
            text="Objetivo: --",
            font_size=dp(14),
            size_hint_y=None, height=dp(25),
            color=(0.6, 0.6, 0.6, 1),
        )
        root.add_widget(self.target_display)

        self.progress = ProgressBar(max=100, value=0, size_hint_y=None, height=dp(22))
        root.add_widget(self.progress)

        self.progress_label = Label(
            text="0.0%", font_size=dp(13),
            size_hint_y=None, height=dp(22),
            color=(0.75, 0.75, 0.75, 1),
        )
        root.add_widget(self.progress_label)

        row1 = BoxLayout(orientation="horizontal", size_hint_y=None, height=dp(46), spacing=dp(8))
        row1.add_widget(Label(text="Cantidad:", font_size=dp(15), size_hint_x=0.28, color=(0.8, 0.8, 0.8, 1)))
        self.target_input = TextInput(
            text="100", input_filter="float", font_size=dp(17),
            multiline=False, size_hint_x=0.35,
            background_color=(0.16, 0.16, 0.2, 1),
            foreground_color=(1, 1, 1, 1),
            cursor_color=(0.3, 0.75, 1, 1),
            padding=[dp(10), dp(8), 0, 0],
        )
        row1.add_widget(self.target_input)
        self.unit_spinner = Spinner(
            text="MB", values=("MB", "GB"),
            font_size=dp(15), size_hint_x=0.37,
            background_color=(0.2, 0.2, 0.26, 1), color=(1, 1, 1, 1),
        )
        row1.add_widget(self.unit_spinner)
        root.add_widget(row1)

        row_mode = BoxLayout(orientation="horizontal", size_hint_y=None, height=dp(46), spacing=dp(8))
        row_mode.add_widget(Label(text="Modo:", font_size=dp(15), size_hint_x=0.28, color=(0.8, 0.8, 0.8, 1)))
        self.mode_spinner = Spinner(
            text="Descarga + Subida", values=("Descarga + Subida", "Solo Descarga", "Solo Subida"),
            font_size=dp(14), size_hint_x=0.72,
            background_color=(0.2, 0.2, 0.26, 1), color=(1, 1, 1, 1),
        )
        row_mode.add_widget(self.mode_spinner)
        root.add_widget(row_mode)

        row2 = BoxLayout(orientation="horizontal", size_hint_y=None, height=dp(46), spacing=dp(8))
        row2.add_widget(Label(text="Hilos:", font_size=dp(15), size_hint_x=0.28, color=(0.8, 0.8, 0.8, 1)))
        self.threads_spinner = Spinner(
            text="4", values=("1", "2", "3", "4", "6", "8", "10", "12"),
            font_size=dp(15), size_hint_x=0.72,
            background_color=(0.2, 0.2, 0.26, 1), color=(1, 1, 1, 1),
        )
        row2.add_widget(self.threads_spinner)
        root.add_widget(row2)

        btn_row = BoxLayout(orientation="horizontal", size_hint_y=None, height=dp(50), spacing=dp(10))
        self.start_btn = Button(
            text="INICIAR", font_size=dp(17),
            background_color=(0.1, 0.65, 0.25, 1), background_normal="",
            color=(1, 1, 1, 1),
        )
        self.start_btn.bind(on_press=self.start_consuming)
        btn_row.add_widget(self.start_btn)

        self.stop_btn = Button(
            text="DETENER", font_size=dp(17),
            background_color=(0.7, 0.15, 0.15, 1), background_normal="",
            color=(1, 1, 1, 1), disabled=True,
        )
        self.stop_btn.bind(on_press=self.stop_consuming)
        btn_row.add_widget(self.stop_btn)
        root.add_widget(btn_row)

        self.reset_btn = Button(
            text="REINICIAR CONTADOR", font_size=dp(13),
            size_hint_y=None, height=dp(40),
            background_color=(0.3, 0.3, 0.35, 1), background_normal="",
            color=(1, 1, 1, 1),
        )
        self.reset_btn.bind(on_press=self.reset_counter)
        root.add_widget(self.reset_btn)

        self.status_label = Label(
            text="Listo", font_size=dp(12),
            size_hint_y=None, height=dp(25),
            color=(0.45, 0.45, 0.45, 1),
        )
        root.add_widget(self.status_label)

        root.add_widget(Label(size_hint_y=1))

        Clock.schedule_interval(self.update_ui, 0.5)
        return root

    def fmt(self, nbytes):
        if nbytes >= 1_073_741_824:
            return f"{nbytes / 1_073_741_824:.2f} GB"
        if nbytes >= 1_048_576:
            return f"{nbytes / 1_048_576:.2f} MB"
        if nbytes >= 1024:
            return f"{nbytes / 1024:.2f} KB"
        return f"{nbytes} B"

    def _add_bytes(self, n):
        with self.lock:
            self.total_bytes += n
            self.speed_tracker_bytes += n

    def _reached_target(self):
        return self.target_bytes > 0 and self.total_bytes >= self.target_bytes

    def start_consuming(self, *_):
        try:
            amount = float(self.target_input.text)
        except ValueError:
            self.status_label.text = "Escribe un numero valido"
            return
        if amount <= 0:
            self.status_label.text = "Debe ser mayor a 0"
            return

        mult = 1_073_741_824 if self.unit_spinner.text == "GB" else 1_048_576
        self.target_bytes = int(amount * mult)
        self.target_display.text = f"Objetivo: {amount} {self.unit_spinner.text}"
        self.running = True

        for w in (self.start_btn, self.target_input, self.unit_spinner, self.threads_spinner, self.mode_spinner):
            w.disabled = True
        self.stop_btn.disabled = False
        self.status_label.text = "Consumiendo datos..."
        self.status_label.color = (0.3, 0.75, 1, 1)

        n_threads = int(self.threads_spinner.text)
        mode = self.mode_spinner.text

        if mode in ("Descarga + Subida", "Solo Descarga"):
            dl_threads = n_threads if mode == "Solo Descarga" else max(1, n_threads // 2)
            for i in range(dl_threads):
                threading.Thread(target=self._download_worker, args=(i,), daemon=True).start()

        if mode in ("Descarga + Subida", "Solo Subida"):
            ul_threads = n_threads if mode == "Solo Subida" else max(1, n_threads - n_threads // 2)
            for i in range(ul_threads):
                threading.Thread(target=self._upload_worker, args=(i,), daemon=True).start()

        threading.Thread(target=self._speed_loop, daemon=True).start()

    def stop_consuming(self, *_):
        self.running = False
        for w in (self.start_btn, self.target_input, self.unit_spinner, self.threads_spinner, self.mode_spinner):
            w.disabled = False
        self.stop_btn.disabled = True
        self.status_label.text = "Detenido"
        self.status_label.color = (0.9, 0.6, 0.1, 1)

    def reset_counter(self, *_):
        if not self.running:
            with self.lock:
                self.total_bytes = 0
            self.progress.value = 0
            self.progress_label.text = "0.0%"
            self.consumed_label.text = "Consumido: 0.00 MB"
            self.status_label.text = "Reiniciado"

    def _download_worker(self, tid):
        idx = tid % len(DOWNLOAD_URLS)
        while self.running and not self._reached_target():
            url = DOWNLOAD_URLS[idx]
            try:
                req = urllib.request.Request(url)
                req.add_header("User-Agent", "Mozilla/5.0 (Linux; Android 12)")
                resp = urllib.request.urlopen(req, timeout=20, context=SSL_CONTEXT)
                while self.running and not self._reached_target():
                    chunk = resp.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    self._add_bytes(len(chunk))
                    del chunk
                resp.close()
            except Exception:
                time.sleep(2)
            idx = (idx + 1) % len(DOWNLOAD_URLS)
        self._check_done()

    def _upload_worker(self, tid):
        idx = tid % len(UPLOAD_URLS)
        random_block = os.urandom(UPLOAD_BLOCK_SIZE)
        while self.running and not self._reached_target():
            url = UPLOAD_URLS[idx]
            try:
                req = urllib.request.Request(url, data=random_block, method="POST")
                req.add_header("User-Agent", "Mozilla/5.0 (Linux; Android 12)")
                req.add_header("Content-Type", "application/octet-stream")
                resp = urllib.request.urlopen(req, timeout=20, context=SSL_CONTEXT)
                resp.read()
                resp.close()
                self._add_bytes(UPLOAD_BLOCK_SIZE)
            except Exception:
                time.sleep(2)
            idx = (idx + 1) % len(UPLOAD_URLS)
        self._check_done()

    def _speed_loop(self):
        while self.running:
            with self.lock:
                self.speed_bytes = self.speed_tracker_bytes
                self.speed_tracker_bytes = 0
            time.sleep(1)
        with self.lock:
            self.speed_bytes = 0

    def _check_done(self):
        if self._reached_target() and self.running:
            Clock.schedule_once(lambda dt: self._on_target_reached())

    def _on_target_reached(self):
        if not self.running:
            return
        self.stop_consuming()
        self.status_label.text = "Objetivo alcanzado!"
        self.status_label.color = (0.2, 0.95, 0.4, 1)

    def update_ui(self, _dt):
        with self.lock:
            total = self.total_bytes
            speed = self.speed_bytes

        self.consumed_label.text = f"Consumido: {self.fmt(total)}"
        self.speed_label.text = f"{(speed * 8) / 1_000_000:.2f} Mbps"

        if self.target_bytes > 0:
            pct = min(100.0, (total / self.target_bytes) * 100)
            self.progress.value = pct
            self.progress_label.text = f"{pct:.1f}%"


if __name__ == "__main__":
    DataConsumerApp().run()

In [ ]:
# Verificar que main.py se creo correctamente
import os
size = os.path.getsize('main.py')
print('=' * 50)
print('PASO 3: Codigo fuente creado')
print('=' * 50)
print(f'Archivo: main.py ({size} bytes)')
print(f'Ubicacion: {os.path.abspath("main.py")}')
print()
print('PASO 3 COMPLETADO')

## 4. Crear buildozer.spec (configuracion del APK)

In [ ]:
%%writefile buildozer.spec
[app]
title = Data Consumer
package.name = dataconsumer
package.domain = org.dataconsumer
source.dir = .
source.include_exts = py,png,jpg,kv,atlas
source.exclude_dirs = .git,.github,bin,.buildozer
version = 1.1.0
requirements = python3,kivy==2.3.0,android,certifi,openssl
orientation = portrait
fullscreen = 0
android.permissions = INTERNET,ACCESS_NETWORK_STATE
android.api = 35
android.minapi = 24
android.ndk = 25b
android.archs = arm64-v8a
android.allow_backup = True
android.accept_sdk_license = True
p4a.branch = develop

[buildozer]
log_level = 2
warn_on_root = 0

In [ ]:
# Verificar que buildozer.spec se creo correctamente
size = os.path.getsize('buildozer.spec')
print('=' * 50)
print('PASO 4: Configuracion del APK creada')
print('=' * 50)
print(f'Archivo: buildozer.spec ({size} bytes)')
print()
print('Archivos en el proyecto:')
for f in os.listdir('.'):
    print(f'  -> {f}')
print()
print('PASO 4 COMPLETADO')

## 5. Compilar el APK
Esta celda tarda **10-15 minutos** la primera vez. Veras los logs del proceso en tiempo real.

In [ ]:
import os
import time
import subprocess
import sys
import threading

os.environ['BUILDOZER_WARN_ON_ROOT'] = '0'

print('=' * 60)
print('PASO 5: COMPILANDO APK')
print('=' * 60)
print()

# --- Fase A: Primer intento de build para que descargue el SDK ---
print('[1/3] Preparando SDK y descargando dependencias...')
print('      (esto puede tardar unos minutos)')
print()

first_run = subprocess.run(
    ['buildozer', 'android', 'debug'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    env={**os.environ, 'BUILDOZER_WARN_ON_ROOT': '0'}
)

# --- Fase B: Aceptar TODAS las licencias del SDK ---
print('[2/3] Aceptando licencias del Android SDK...')
sdk_path = os.path.expanduser('~/.buildozer/android/platform/android-sdk')
sdkmanager = os.path.join(sdk_path, 'tools', 'bin', 'sdkmanager')

# Buscar sdkmanager en varias ubicaciones posibles
for candidate in [
    os.path.join(sdk_path, 'tools', 'bin', 'sdkmanager'),
    os.path.join(sdk_path, 'cmdline-tools', 'latest', 'bin', 'sdkmanager'),
    os.path.join(sdk_path, 'cmdline-tools', 'bin', 'sdkmanager'),
]:
    if os.path.exists(candidate):
        sdkmanager = candidate
        break

if os.path.exists(sdkmanager):
    result = subprocess.run(
        f'yes | {sdkmanager} --licenses --sdk_root={sdk_path}',
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    print('      Licencias aceptadas OK')
else:
    # Metodo alternativo: crear los archivos de licencia manualmente
    print('      sdkmanager no encontrado, aceptando licencias manualmente...')
    licenses_dir = os.path.join(sdk_path, 'licenses')
    os.makedirs(licenses_dir, exist_ok=True)

    license_hash = '\n8933bad161af4178b1185d1a37fbf41ea5269c55\nd56f5187479451eabf01fb78af6dfcb131a6481e\n24333f8a63b6825ea9c5514f83c2829b004d1fee'
    for lic_file in ['android-sdk-license', 'android-sdk-preview-license', 'android-googletv-license',
                     'google-gdk-license', 'intel-android-extra-license', 'android-sdk-arm-dbt-license']:
        with open(os.path.join(licenses_dir, lic_file), 'w') as f:
            f.write(license_hash)

    print('      Licencias escritas manualmente OK')

# --- Fase C: Compilar de nuevo ---
print()
print('[3/3] Compilando APK (esto tarda 10-15 minutos)...')
print()

start_time = time.time()
last_phase = ['']

done_event = threading.Event()

def timer_thread():
    while not done_event.is_set():
        elapsed = time.time() - start_time
        mins = int(elapsed // 60)
        secs = int(elapsed % 60)
        if elapsed >= 30:
            print(f'\n--- Tiempo transcurrido: {mins}m {secs}s ---', flush=True)
        done_event.wait(30)

t = threading.Thread(target=timer_thread, daemon=True)
t.start()

process = subprocess.Popen(
    ['buildozer', 'android', 'debug'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    env={**os.environ, 'BUILDOZER_WARN_ON_ROOT': '0'}
)

keywords = [
    'Download', 'download', 'Compil', 'compil', 'Build', 'build',
    'Install', 'install', 'Extract', 'extract', 'Unpack', 'unpack',
    'Running', 'running', 'Copy', 'copy', 'Creat', 'creat',
    'Pack', 'pack', 'Sign', 'sign', 'Gradle', 'gradle',
    'APK', 'apk', 'SUCCESS', 'ERROR', 'error', 'FAIL', 'fail',
    'python-for-android', 'p4a', 'sdk', 'SDK', 'ndk', 'NDK',
    'Distribut', 'distribut', 'Recipe', 'recipe',
]

line_count = 0
for line in process.stdout:
    line = line.rstrip()
    line_count += 1

    if any(kw in line for kw in keywords):
        if 'Downloading' in line or 'Download' in line:
            phase = 'DESCARGANDO'
        elif 'Compil' in line or 'compil' in line or 'Build' in line:
            phase = 'COMPILANDO'
        elif 'Gradle' in line or 'gradle' in line:
            phase = 'GRADLE BUILD'
        elif 'Pack' in line or 'Sign' in line:
            phase = 'EMPAQUETANDO'
        else:
            phase = ''

        if phase and phase != last_phase[0]:
            elapsed = time.time() - start_time
            mins = int(elapsed // 60)
            secs = int(elapsed % 60)
            print(f'\n{"=" * 40}')
            print(f'FASE: {phase} ({mins}m {secs}s)')
            print(f'{"=" * 40}')
            last_phase[0] = phase

        display = line[:120] + '...' if len(line) > 120 else line
        print(f'  {display}', flush=True)

    elif 'error' in line.lower() or 'exception' in line.lower() or 'traceback' in line.lower():
        print(f'  [!] {line}', flush=True)

process.wait()
done_event.set()

elapsed = time.time() - start_time
mins = int(elapsed // 60)
secs = int(elapsed % 60)

print()
print('=' * 60)
if process.returncode == 0:
    print(f'COMPILACION EXITOSA! Tiempo total: {mins}m {secs}s')
    print(f'Lineas de log procesadas: {line_count}')
else:
    print(f'ERROR EN COMPILACION (codigo: {process.returncode})')
    print(f'Tiempo transcurrido: {mins}m {secs}s')
    print('Revisa los logs de arriba para ver el error.')
print('=' * 60)

## 6. Descargar el APK

In [ ]:
import glob
import os
from google.colab import files

print('=' * 50)
print('PASO 6: Descargando APK...')
print('=' * 50)
print()

apk_files = glob.glob('/content/dataconsumer/bin/*.apk')

if apk_files:
    apk_path = apk_files[0]
    apk_name = os.path.basename(apk_path)
    apk_size = os.path.getsize(apk_path) / 1_048_576

    print(f'APK encontrado!')
    print(f'  Nombre:  {apk_name}')
    print(f'  Tamanio: {apk_size:.1f} MB')
    print(f'  Ruta:    {apk_path}')
    print()
    print('Iniciando descarga...')
    files.download(apk_path)
    print()
    print('LISTO! Pasa el APK a tu celular e instalalo.')
else:
    print('ERROR: No se encontro el APK en /content/dataconsumer/bin/')
    print()
    print('Archivos disponibles:')
    bin_path = '/content/dataconsumer/bin'
    if os.path.exists(bin_path):
        for f in os.listdir(bin_path):
            print(f'  -> {f}')
    else:
        print('  La carpeta bin/ no existe. La compilacion fallo.')
        print('  Vuelve a ejecutar el Paso 5 y revisa los errores.')